# Find a chemical structure in your documents

Upload PDFs / Word files, give a SMILES, and get back **which document, which
page, and where on the page** the structure is drawn.

Pipeline (PatCID, *Nature Communications* 15:6532, 2024):
**DECIMER-Segmentation** → **MolClassifier** → **DECIMER / MolGrapher** → **RDKit matching**

Models and the per-document extraction cache are kept on your Google Drive, so
the second time you open this notebook nothing is re-downloaded and documents
you already processed are searched instantly.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Everything
works on CPU too, just several times slower (a 50-page patent is minutes on GPU,
tens of minutes on CPU).

## 1. Check the runtime

In [ ]:
#@title Runtime check — the Python version decides which engines you can use
import sys, subprocess

print("Python:", sys.version.split()[0])
try:
    print(subprocess.check_output(["nvidia-smi", "-L"], text=True).strip())
except Exception:
    print("GPU: none (CPU-only run — this works, just slower)")

if sys.version_info[:2] != (3, 11):
    print(
        "\nNOTE: MolGrapher pins its torch wheels to CPython 3.11, so on this\n"
        "runtime use the DECIMER recognition engine (section 2). It is the\n"
        "engine PatCID benchmarked at 67.2% vs MolGrapher's 63.0% on D2C-RND,\n"
        "so you lose nothing on accuracy — only some CPU speed."
    )

## 2. Install

Two pip installs and one apt install. This is the DECIMER path: pure pip, no
version pinning problems, works on any current Colab runtime.

In [ ]:
#@title Install (~5 min on a fresh runtime)
# core: matching, PDF/DOCX handling, reporting
!pip install -q rdkit pymupdf python-docx

# segmentation + recognition (both TensorFlow-based, both pip-installable)
!pip install -q decimer-segmentation decimer

# .docx -> PDF so Word documents keep their real page numbers
!apt-get -qq install -y libreoffice-writer > /dev/null


In [ ]:
#@title Verify the install actually works (fails fast if it does not)
import DECIMER, decimer_segmentation, rdkit, pymupdf
print("DECIMER               ", DECIMER.__file__.split('/site-packages/')[-1])
print("decimer-segmentation  ", decimer_segmentation.__version__)
print("rdkit                 ", rdkit.__version__)
print("pymupdf               ", pymupdf.__doc__.split()[1])
print("\nOK — importing DECIMER above also downloaded its model weights.")

### Optional: MolClassifier (Clean / Markush / Trash)

Recommended. It filters segmentation errors and tells you which detections are
**Markush** structures (generic structures with R groups — those can never equal
a concrete SMILES, so you want them flagged rather than silently missed).

torch and torchvision are already on Colab, so this is a small install.

In [ ]:
#@title Install MolClassifier
!git clone -q https://github.com/DS4SD/MolClassifier.git /content/MolClassifier
!pip install -q pycocotools albumentations imantics more-itertools

### Optional: MolGrapher instead of DECIMER — **Python 3.11 runtimes only**

MolGrapher is the engine PatCID itself used (about 2× faster than DECIMER on
CPU). Its `setup.py` builds torch wheel URLs pinned to CPython 3.11, so on a
3.12+ runtime this cell **will fail** — that is expected, skip it and stay on
DECIMER. Run the cell in section 1 first to see which you have.

In [ ]:
#@title Install MolGrapher (skip unless Python is 3.11)
import sys
if sys.version_info[:2] == (3, 11):
    !git clone -q https://github.com/DS4SD/MolGrapher.git /content/MolGrapher
    !cd /content/MolGrapher && pip install -q -e ".[cpu]" && bash install_paddleocr.sh
    print("MolGrapher installed — you can set RECOGNIZER = 'molgrapher' below.")
else:
    print(f"Skipped: Python {sys.version_info.major}.{sys.version_info.minor} "
          "is not 3.11. Stay on RECOGNIZER = 'decimer'.")

### Get the tool itself

In [ ]:
#@title Clone structure_finder and register the import paths
!git clone -q --branch claude/structure-identification-documents-0kwz8a https://github.com/ranjitranbhor/PatCID.git /content/PatCID

In [ ]:
#@title Set up sys.path
import os, sys

paths = ["/content/PatCID"]
if os.path.isdir("/content/MolClassifier"):
    # MolClassifier imports `albumentations_transforms` as a TOP-LEVEL module
    # even though the file lives inside the package, so its inner directory has
    # to be on the path as well as the repo root.
    paths += ["/content/MolClassifier", "/content/MolClassifier/mol_classifier"]

for path in paths:
    if path not in sys.path:
        sys.path.insert(0, path)

import structure_finder
print("structure_finder", structure_finder.__version__)
print("import paths:", paths)

In [ ]:
#@title Smoke test — 28 tests, ~10 s, no model weights needed
!cd /content/PatCID && python -m pytest tests/test_structure_finder.py -q

## 3. Mount Google Drive

Everything heavy lands in `MyDrive/structure_finder/`:

```
models/        MolClassifier checkpoint
models/hf/     HF_HOME      — MolGrapher weights
models/pystow/ PYSTOW_HOME  — DECIMER weights
cache/         per-document extraction results   <- the valuable part
outputs/       reports and annotated pages
```

The cache is keyed by file hash, so a document you processed last week is never
re-processed — searching a different molecule across it takes milliseconds.

In [ ]:
#@title Mount Drive and pre-download the model weights
from structure_finder import resolve_workspace

workspace = resolve_workspace(use_drive=True)
print("Workspace:", workspace.root)

!python -m structure_finder.setup_models --drive --engine molclassifier --engine decimer-seg

## 4. Upload your documents

Accepts `.pdf`, `.docx`, `.doc` and image files. You can also skip this cell and
point the search at a Drive folder instead — set `DOCUMENTS` in section 6.

In [ ]:
#@title Upload PDFs / Word documents
import os, shutil
from google.colab import files

UPLOAD_DIR = "/content/documents"
os.makedirs(UPLOAD_DIR, exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join(UPLOAD_DIR, name))

print(f"\n{len(os.listdir(UPLOAD_DIR))} document(s) in {UPLOAD_DIR}:")
for name in sorted(os.listdir(UPLOAD_DIR)):
    print("  ", name)

## 5. Enter the structure you are looking for

In [ ]:
#@title Query structure
QUERY_SMILES = "CC(=O)Oc1ccccc1C(=O)O"  #@param {type:"string"}

from rdkit import Chem
from rdkit.Chem import Draw

molecule = Chem.MolFromSmiles(QUERY_SMILES)
assert molecule is not None, "That SMILES could not be parsed — check it."
Chem.RemoveStereochemistry(molecule)
print("Canonical (no stereo):", Chem.MolToSmiles(molecule))
print("InChIKey (no stereo): ", Chem.MolToInchiKey(molecule))
Draw.MolToImage(molecule, size=(350, 350))

## 6. Search

The first pass over a document is the slow one — every page is segmented,
classified and read. That work is query-independent and cached on Drive, so
section 9 (a different molecule, same documents) returns immediately.

In [ ]:
#@title Run the search
import os
from structure_finder import find_structure, format_summary

DOCUMENTS  = [UPLOAD_DIR]        #@param — or e.g. ["/content/drive/MyDrive/patents"]
RECOGNIZER = "decimer"           #@param ["decimer", "molgrapher", "ensemble"]
DPI        = 300                 #@param {type:"integer"}

results = find_structure(
    documents=DOCUMENTS,
    smiles=QUERY_SMILES,
    use_drive=True,
    segmenter="decimer",                                  # DECIMER-Segmentation
    classifier="molclassifier" if os.path.isdir("/content/MolClassifier") else "none",
    recognizer=RECOGNIZER,
    match_modes=("exact", "connectivity", "tautomer"),
    dpi=DPI,
)

print(format_summary(results))

## 7. See the matches in context

In [ ]:
#@title Write the report and show the annotated pages
from pathlib import Path
from IPython.display import Image, display
from structure_finder import save_all

output_dir = Path(workspace.outputs) / "latest"
written = save_all(results, output_dir, annotate=True)
print("Report:", output_dir, "\n")

pages = written.get("annotated_pages", [])
for page_path in pages:
    print(page_path)
    display(Image(filename=page_path, width=760))

if not pages:
    print("No image hits to annotate — see the summary above, and section 10.")

In [ ]:
#@title Hits as a table (and download the CSV)
import pandas as pd

hits = pd.DataFrame(results["hits"])
display(hits if len(hits) else "No matches.")

if len(hits):
    from google.colab import files
    hits.to_csv("/content/hits.csv", index=False)
    files.download("/content/hits.csv")

## 8. Everything the pipeline read

`structure_search_extractions.jsonl` lists **every** chemical image found — page,
bounding box, class and SMILES — whether or not it matched. This is what you
check when a search comes back empty: it tells you whether the depiction was
missed by the segmenter or misread by the recognizer.

In [ ]:
#@title Browse all recognised structures
import json, pandas as pd

rows = []
for line in open(output_dir / "structure_search_extractions.jsonl"):
    record = json.loads(line)
    for figure in record["figures"]:
        rows.append({
            "document": record["document"]["filename"],
            "page": figure["page"],
            "class": figure["figure_class"],
            "smiles": figure["canonical_smiles"],
            "confidence": figure["recognition_confidence"],
        })

everything = pd.DataFrame(rows)
print(f"{len(everything)} chemical image(s) found")
display(everything.head(50))

## 9. Search another structure — fast, the cache is already warm

In [ ]:
#@title Another query over the same documents
SECOND_QUERY = "Cn1cnc2c1c(=O)n(C)c(=O)n2C"  #@param {type:"string"}

results_2 = find_structure(
    documents=DOCUMENTS,
    smiles=SECOND_QUERY,
    use_drive=True,
    recognizer=RECOGNIZER,
    match_modes=("exact", "connectivity", "tautomer"),
)
print(format_summary(results_2))

## 10. Reading the result honestly

From the PatCID paper (Table 3), the full segment → classify → recognise chain
scores **54.5% precision / 46.0% recall** on its random benchmark and
**41.3% / 44.5%** on the deliberately hard one (older documents, non-US patent
offices, less standard drawing styles).

So: **a hit is strong evidence, a miss is weak evidence.**

If you expected a hit and got none, widen the search before concluding anything:

```python
results = find_structure(
    documents=DOCUMENTS,
    smiles=QUERY_SMILES,
    use_drive=True,
    recognizer="ensemble",          # DECIMER + MolGrapher, keeps the better read
    match_modes=("exact", "connectivity", "tautomer", "similarity"),
    similarity_threshold=0.85,      # catches small recognition errors
    dpi=400,                        # small or low-quality depictions
)
```

then look at section 8 to see what the pipeline actually read.

**Markush structures** (generic structures with R groups) have no single molecule
behind them, so they can never match a concrete SMILES. They are counted
separately in the summary. Search their scaffold instead:

```python
find_structure(documents=DOCUMENTS, smiles="smarts:c1ccc2[nH]ccc2c1",
               match_modes=("substructure",), use_drive=True)
```

**Word documents** keep real page numbers only because LibreOffice is installed
in section 2. Without it, the tool falls back to the images embedded in the
`.docx` and numbers them sequentially.